In [236]:
import torch
from math import sqrt

In [97]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [98]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [99]:
text[:8]
encoded_text = torch.tensor(encode(text), dtype = torch.long)

In [706]:
x, y = get_batch(train_data)

In [397]:
vocab_size = len(chars)
d_model = 10
embedding = torch.nn.Embedding(vocab_size, d_model)

In [398]:
vocab_size

110

In [710]:

class Head(torch.nn.Module):

    def __init__(self, block_size, d_model, d_v, d_k):
        super().__init__()
        self.d_k = d_k
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_v)
        self.register_buffer("causal_mask", self.ins_causal_mask(block_size))


    def forward(self, X_e):
        B, T, d_model = X_e.shape
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_v)
        QK = Q @ K.transpose(1,2)
        QK_scale = QK/math.sqrt(self.d_k) #(B,T,T)
        QK_scale += self.causal_mask[:T, :T]
        QV_softmax = torch.nn.functional.softmax(QK_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V #(B, T, d_v)
        return added_weights


    @staticmethod
    def ins_causal_mask(block_size):
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


class MLP(torch.nn.Module):

    def __init__(self, d_model):
            
        super().__init__()
        self.W_ff1 = torch.nn.Linear(d_model, 4*d_model)
        self.W_ff2 = torch.nn.Linear(4*d_model, d_model)

    def forward(self, X):

        linear_1 = self.W_ff1(X)
        non_linearity = torch.nn.functional.gelu(linear_1)
        linear_2 = self.W_ff2(non_linearity)
        return linear_2

        

class MultiHead(torch.nn.Module):
    
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        assert d_model % n_heads == 0
        d_v = d_model // n_heads
        d_k = d_k if d_k is not None else d_v
        self.heads = torch.nn.ModuleList([Head(block_size, d_model, d_v, d_k) for _ in range(n_heads)])
        self.W_o = torch.nn.Linear(d_model, d_model)

    def forward(self, X_e):

        added_weights = [head(X_e) for head in self.heads]
        concat = torch.cat(added_weights, dim = -1)
        return self.W_o(concat)

class Block(torch.nn.Module):
     
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        self.att = MultiHead(block_size, d_model, n_heads, d_k)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.mlp = MLP(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)

    def forward(self, X):

        X = X + self.att(self.ln_1(X))
        X = X + self.mlp(self.ln_2(X))
        return X


class decoder_only_transformer(torch.nn.Module):

    def __init__(self, vocab_size, block_size, d_model, n_heads = 1, d_k = None, n_layers = 1):

        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model) 
        self.blocks = torch.nn.ModuleList([Block(block_size, d_model, n_heads, d_k) for _ in range(n_layers)])
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)
        self.register_buffer("pos_encoding", self.sinusoidal_pe(block_size, d_model))
        self.loss = torch.nn.CrossEntropyLoss()

    
    @staticmethod
    def sinusoidal_pe(block_size, d_model):
        pe = torch.zeros(block_size, d_model)
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denom)
                pe[pos, i + 1] = math.cos(pos / denom)
        return pe

    
    
    def forward(self, X, Y = None):

        B, T = X.shape
        X_e = self.embedding(X) + self.pos_encoding[:T] #(B,T,d_model)
        for block in self.blocks:
            X_e = block(X_e)
        X_e = self.ln_f(X_e) #(B,T,d_model)
        logits = self.linear(X_e) #(B,T,vocab_size)
        loss = None
        if Y is not None:
            loss = self.loss(logits.reshape(B*T,-1), Y.reshape(B*T,))
        
        return logits, loss



In [ ]:
vocab_size = len(chars)
block_size = 8

model = decoder_only_transformer(
    vocab_size=vocab_size,
    block_size=block_size,
    d_model=32,
    n_heads=4,
    n_layers=4,
)
logits, loss = model(x, y);  print(logits.shape, loss.item(), math.log(vocab_size))
logits, loss = model(x);     print(logits.shape, loss)      # loss → None

torch.Size([1, 8, 110]) 4.726049423217773 4.700480365792417
torch.Size([1, 8, 110]) None


In [718]:
train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 8
batch_size = 1
torch.manual_seed(0)

def get_batch(data, batch_size, block_size):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [723]:
@torch.no_grad()
def estimate_loss(modelo, eval_iters=50):
    modelo.eval()
    out = {}
    for split, data in [("train", train_data), ("val", test_data)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(data, batch_size, block_size)
            _, loss = modelo(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    modelo.train()
    return out

In [725]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
max_steps = 100000
eval_interval = 1000
for step in range(max_steps):
    if step % eval_interval == 0:
        losses = estimate_loss(model)
        print(f"step {step}: train {losses['train']:.3f}, val {losses['val']:.3f}")

    x, y = get_batch(train_data, batch_size, block_size)
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train 4.723, val 4.709
step 1000: train 2.792, val 2.626
step 2000: train 2.492, val 2.310
step 3000: train 2.386, val 2.295
step 4000: train 2.294, val 2.109
step 5000: train 2.181, val 2.133
step 6000: train 2.258, val 2.101
step 7000: train 2.084, val 2.068
step 8000: train 2.105, val 2.210
step 9000: train 2.163, val 1.968
step 10000: train 2.120, val 2.088
step 11000: train 2.167, val 2.013
step 12000: train 2.112, val 2.016
step 13000: train 2.070, val 1.989
step 14000: train 2.033, val 2.099
step 15000: train 2.080, val 1.900
step 16000: train 2.010, val 2.113
step 17000: train 1.979, val 1.774
step 18000: train 1.925, val 1.978
step 19000: train 1.957, val 2.120
step 20000: train 2.020, val 1.929
step 21000: train 1.892, val 2.106
step 22000: train 2.004, val 1.973
step 23000: train 2.085, val 2.065
step 24000: train 2.041, val 1.915
step 25000: train 1.894, val 1.947
step 26000: train 1.924, val 1.938
step 27000: train 2.053, val 1.921
step 28000: train 1.914, val 1.98

In [421]:
decode(clases_predichas.tolist()[0])

['X', '¡', '\\', 'U', 'a', 'a', 'Z', 'e']

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(first_batch[0][0], first_batch[1][0])
print(f'Total loss for this batch: {loss.item()}')

In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)